Begin by importing necessary packages

In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, regexp_replace, explode, array, struct, coalesce, when, trim
from pyspark.sql.types import StructType, IntegerType # Need this for the empty DataFrame above

Set global variables, define filepaths, and create lists of variables

In [40]:
spark = SparkSession.builder \
    .appName("CampusSafetyTrendAnalysis") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .getOrCreate()

hdfs_write_root = "hdfs://localhost:9000/data/merged"

# All years present in the dataset (from 2018 to 2023)
ALL_YEARS = ["18", "19", "20", "21", "22", "23"]

# The base institution fields to keep
INSTITUTION_FIELDS = ["UNITID_P", "INSTNM", "OPEID", "BRANCH", "Address", "City", "State", "ZIP", "sector_cd", "Sector_desc", "men_total", "women_total", "Total"]

# Mapping of all possible offense types across all data files
OFFENSE_MAP = {
    "crime": ["MURD", "NEG_M", "RAPE", "FONDL", "INCES", "STATR", "ROBBE", "AGG_A", "BURGLA", "VEHIC", "ARSON"],
    "discipline": ["WEAPON", "DRUG", "LIQUOR"],
    "vawa": ["DOMEST", "DATING", "STALK"],
    "hate": ["MURD", "RAPE", "FOND", "INCE", "STAT", "ROBBE", "AGG_A", "BURGLA", "VEHIC", "ARSON", "SIM_A", "LAR_T", "INTIM", "VANDAL"],
    # Suffixes for hate crimes are complex, so we'll handle them inside the function
}

Begin by loading data

In [41]:

try:
    crime_df = spark.read.parquet(f"{hdfs_write_root}/crime.parquet")
    hate_df = spark.read.parquet(f"{hdfs_write_root}/hate.parquet")
    discipline_df = spark.read.parquet(f"{hdfs_write_root}/discipline.parquet")
    vawa_df = spark.read.parquet(f"{hdfs_write_root}/vawa.parquet")

except Exception as e:
    print(f"Error loading parquet: {e}")
    spark.stop()

Function to transfrom "wide" format (e.g., MURD18, MURD19, etc.) into a "long" format, where we have a single row per institution, per year, per location, and per offense type.

In [42]:
def transform_to_long_format(df, data_type, location):
    """
    Transforms a single wide-format DataFrame (e.g., oncampuscrime181920_df)
    into a long-format DataFrame with unique columns for Year, Location, Offense Type, and Count.
    Much of this function was generated by Gemini.
    """
    # 1. Determine offense codes based on the data type
    offense_codes = OFFENSE_MAP.get(data_type, []) # OFFENSE_MAP is a dictionary.
    # default returns a [] if empty.
    #https://www.w3schools.com/python/ref_dictionary_get.asp

    # Special handling for Hate Crime Offenses and Suffixes. Hate Crime header is much more complex than others
    if data_type == "hate":
        # Get all non-institution columns that don't start with FILTER
        # This dynamically captures all offense+suffix+year combinations
        offense_columns = [c for c in df.columns if c not in INSTITUTION_FIELDS and not c.startswith("FILTER")]
        # Create an array of (Offense_Year, Value) pairs
        kvs = [
            struct(lit(regexp_replace(c, r'(\d{2})$', '')).alias("Offense_Year"), col(c).alias("Count"))
            for c in offense_columns
        ]
    else:
        # For Crime, VAWA, Discipline, create an array of (OffenseType + Year, Value) pairs
        kvs = []
        for year in ALL_YEARS:
            for code in offense_codes:
                col_name = f"{code}{year}"
                if col_name in df.columns:
                    kvs.append(
                        struct(lit(col_name).alias("Offense_Year"), col(col_name).alias("Count"))
                    )

    if not kvs:
        print(f"Warning: No offense columns found for {location} {data_type}. Skipping.")
        return spark.createDataFrame([], StructType([])) # Return an empty DataFrame

    # 2. Explode the array to create a long format
    df_long = df.select(
        *INSTITUTION_FIELDS,
        explode(array(*kvs)).alias("key_value")
    ).select(
        *INSTITUTION_FIELDS,
        col("key_value.Offense_Year").alias("Offense_Year"),
        col("key_value.Count").alias("Count")
    ).withColumn("Location", lit(location))

    # 3. Separate Offense Type and Year
    df_final = df_long.withColumn(
        "Year",
        col("Offense_Year").substr(-2, 2) # Extract the last two characters (the year)
    ).withColumn(
        "Offense_Code",
        # Remove the two-digit year from the end to get the code (e.g., MURD18 -> MURD)
        regexp_replace(col("Offense_Year"), r'\d{2}$', '') # replace last two digits at the end of the string
    ).drop("Offense_Year") # Drop the temporary combined column

    # Convert the two-digit year to a four-digit year for clarity
    df_final = df_final.withColumn(
        "Year",
        col("Year").try_cast(IntegerType()) + lit(2000)
    )

    # Convert count to IntegerType and filter out nulls/zeros if desired (optional)
    df_final = df_final.withColumn(
        "Year",
        col("Year").try_cast(IntegerType()) + lit(2000)
    ).withColumn(
        # FIX: Explicitly handle empty string ('') by setting it to 0 before casting.
        "Count",
        when(
            trim(col("Count")) == lit(""), # Check for empty or whitespace string
            lit(0)                       
        ).otherwise(
            coalesce(col("Count"), lit(0)) # Handle explicit NULLs
        ).try_cast(IntegerType()) # Now safe to cast
    ).filter(col("Count") > 0) # Optionally filter zero-counts, but the casting is safe now.

    return df_final.filter(col("Count") > 0) # Clean the data, remove all zeros

In [43]:
merged_files = [
    ("crime", "ALL_LOCATIONS", f"{hdfs_write_root}/crime.parquet"),
    ("vawa", "ALL_LOCATIONS", f"{hdfs_write_root}/vawa.parquet"),
    ("discipline", "NON_ONCAMPUS", f"{hdfs_write_root}/discipline.parquet"), 
    ("hate", "ALL_LOCATIONS", f"{hdfs_write_root}/hate.parquet"),
]

long_dfs = []
for data_type, location_prefix, path in merged_files:
    print(f"\nProcessing {data_type.upper()} data from {path}...")
    try:
        # Read the merged Parquet file from HDFS
        wide_df = spark.read.parquet(path)
        
        # Transform the wide-format merged DF into the long format
        long_df = transform_to_long_format(wide_df, data_type, location_prefix)
        
        if not long_df.isEmpty():
            long_dfs.append(long_df)
            print(f"Successfully converted {data_type.upper()} to long format.")
            long_dfs[-1].show(5) # https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.show.html
        else:
            print(f"Warning: {data_type.upper()} resulted in an empty long DataFrame.")

    except Exception as e:
        print(f"Error reading or processing {path}: {e}")


Processing CRIME data from hdfs://localhost:9000/data/merged/crime.parquet...
Successfully converted CRIME to long format.
+---------+--------------------+------+-----------+----------------+------+-----+-----+---------+--------------------+---------+-----------+-----+-----+-------------+----+------------+
| UNITID_P|              INSTNM| OPEID|     BRANCH|         Address|  City|State|  ZIP|sector_cd|         Sector_desc|men_total|women_total|Total|Count|     Location|Year|Offense_Code|
+---------+--------------------+------+-----------+----------------+------+-----+-----+---------+--------------------+---------+-----------+-----+-----+-------------+----+------------+
|100654001|Alabama A & M Uni...|100200|Main Campus|4900 MERIDIAN ST|NORMAL|   AL|35762|        1|Public, 4-year or...|     2671|       3943| 6614|    6|ALL_LOCATIONS|4023|        MURD|
|100654001|Alabama A & M Uni...|100200|Main Campus|4900 MERIDIAN ST|NORMAL|   AL|35762|        1|Public, 4-year or...|     2671|       3

In [44]:
if long_dfs:
    # Union all the long-format DataFrames into a single master table
    master_df = long_dfs[0]
    for df in long_dfs[1:]:
        # Use unionByName to ensure schemas align even if columns are in different orders
        master_df = master_df.unionByName(df, allowMissingColumns=True)

    print("\n✅ Final Unified Long-Format Master Table Created.")
    master_df.printSchema()
    master_df.show(5)
    m = master_df.na.drop() # https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropna.html
    m = master_df.na.replace('',None)
    

    # Example Trend: Total Crime Count by Year
    from pyspark.sql.functions import sum as spark_sum
    trend_result = m.groupBy("Year", "Offense_Code","Year").agg(
        spark_sum("Count").alias("TotalOffenses")
    ).orderBy("Offense_Code", "Year")

    print("\n--- Aggregated Trend (Yearly Total Offenses by Type) ---")
    trend_result.show()

else:
    print("❌ Failed to create the final master DataFrame from HDFS data.")

spark.stop()


✅ Final Unified Long-Format Master Table Created.
root
 |-- UNITID_P: long (nullable = true)
 |-- INSTNM: string (nullable = true)
 |-- OPEID: string (nullable = true)
 |-- BRANCH: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP: string (nullable = true)
 |-- sector_cd: integer (nullable = true)
 |-- Sector_desc: string (nullable = true)
 |-- men_total: integer (nullable = true)
 |-- women_total: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Count: integer (nullable = false)
 |-- Location: string (nullable = false)
 |-- Year: integer (nullable = true)
 |-- Offense_Code: string (nullable = true)

+---------+--------------------+------+-----------+----------------+------+-----+-----+---------+--------------------+---------+-----------+-----+-----+-------------+----+------------+
| UNITID_P|              INSTNM| OPEID|     BRANCH|         Address|  City|State|  ZIP|sec

25/12/01 20:20:34 WARN DAGScheduler: Broadcasting large task binary with size 1004.3 KiB
25/12/01 20:20:37 WARN DAGScheduler: Broadcasting large task binary with size 1195.4 KiB


+----+------------+----+-------------+
|Year|Offense_Code|Year|TotalOffenses|
+----+------------+----+-------------+
|NULL|        NULL|NULL|           95|
|4001|           1|4001|       209174|
|4002|           2|4002|          798|
|4003|           3|4003|          360|
|4004|           4|4004|          276|
|4005|           5|4005|          130|
|4006|           6|4006|           18|
|4007|           7|4007|           35|
|4008|           8|4008|            8|
|4009|           9|4009|           36|
|4020|       AGG_A|4020|          545|
|4023|       AGG_A|4023|        13555|
|4019|       ARSON|4019|            4|
|4020|       ARSON|4020|        43516|
|4022|       ARSON|4022|           63|
|4023|       ARSON|4023|        40476|
|4020|      BURGLA|4020|        41000|
|4023|      BURGLA|4023|          812|
|4019|      DATING|4019|         2632|
|4020|      DATING|4020|        42452|
+----+------------+----+-------------+
only showing top 20 rows


In [46]:
# Stop Spark session
spark.stop()